# Medical Device Failure & Healthcare Financial Risk BI Project

## Notebook 02: Data Cleaning

This notebook cleans the raw medical device failure and financial datasets. <br>
Cleaning steps include standardizing column names, converting dates, correcting financial categories, <br>
flagging data quality issues, creating operational risk fields, and exporting cleaned CSV files for SQL, Excel, and Power BI.

In [2]:
# ==========================================
# Notebook 02: Data Cleaning
# Medical Device Failure & Healthcare Financial Risk BI Project
# ==========================================

import pandas as pd
import numpy as no
from pathlib import Path

#Display settings
pd.set_option("display.max_columns",None)
pd.set_option("display.width",120)

print("Libraries imported successfully.")


Libraries imported successfully.


In [3]:
# ==========================================
# Set project folder paths
# ==========================================

PROJECT_ROOT = Path(r"C:\Users\Charl\OneDrive\Documents\GitHub\Medical Device Failure & Healthcare Supply Chain Risk BI Project")

from pathlib import Path

raw_data_path = Path(
    r"C:\Users\Charl\OneDrive\Documents\GitHub\Medical Device Failure & Healthcare Supply Chain Risk BI Project\data\raw"
)

raw_files = [
    raw_data_path / "financial_data.csv",
    raw_data_path / "Medical_Device_Failure_dataset.csv"
]

raw_files 

[WindowsPath('C:/Users/Charl/OneDrive/Documents/GitHub/Medical Device Failure & Healthcare Supply Chain Risk BI Project/data/raw/financial_data.csv'),
 WindowsPath('C:/Users/Charl/OneDrive/Documents/GitHub/Medical Device Failure & Healthcare Supply Chain Risk BI Project/data/raw/Medical_Device_Failure_dataset.csv')]

In [4]:
financial_df = pd.read_csv(raw_files[0])
medical_device_df=pd.read_csv(raw_files[1])

In [5]:
print("Financial data shape:", financial_df.shape)
print("Medical device data shape:", medical_device_df.shape)

Financial data shape: (500, 4)
Medical device data shape: (4149, 13)


In [6]:
# ==========================================
# Create working copies
# ==========================================

device_clean = medical_device_df.copy()
financial_clean = financial_df.copy()


print("Working copies created.")


Working copies created.


In [7]:
# ==========================================
# Standardize column names
# ==========================================

def clean_column_names(df):

    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace(r"[^a-z0-9_]", "", regex=True)

  )
    return df

device_clean    = clean_column_names(device_clean)
financial_clean = clean_column_names(financial_clean)

print("Device columns:")
print(device_clean.columns.tolist())

print("\nFinancial columns:")
print(financial_clean.columns.tolist())

Device columns:
['device_id', 'device_type', 'purchase_date', 'age', 'manufacturer', 'model', 'country', 'maintenance_cost', 'downtime', 'maintenance_frequency', 'failure_event_count', 'maintenance_class', 'maintenance_report']

Financial columns:
['date', 'expense_category', 'amount', 'description']


In [8]:
device_clean      ["purchase_date"] = pd.to_datetime(device_clean["purchase_date"], errors="coerce")
financial_clean    ["date"]         = pd.to_datetime(financial_clean["date"],       errors="coerce") 

print("Device purchase date range:")
print(device_clean["purchase_date"].min(), "to", device_clean["purchase_date"].max()) 

print("\nFinancial date range:")
print(financial_clean["date"].min(), "to", financial_clean["date"].max())


      



Device purchase date range:
2013-04-13 00:00:00 to 2024-04-11 00:00:00

Financial date range:
2024-10-01 00:00:00 to 2026-02-12 00:00:00


In [9]:
# ==========================================
# Clean text fields
# ==========================================

device_text_cols = [
    "device_id",
    "device_type",
    "manufacturer",
    "model",
    "country",
    "maintenance_report"
]

financial_text_cols = [
    "expense_category",
    "description"
]


for col  in device_text_cols:
    device_clean[col] = device_clean[col].astype(str).str.strip()

for col in financial_text_cols:
    financial_clean[col] = financial_clean[col].astype(str).str.strip()

    print("Text fields cleaned")

Text fields cleaned
Text fields cleaned


In [11]:
# ==========================================
# Clean numeric fields
# ==========================================
import numpy as np


# Preserve original maintenance cost
device_clean["maintenance_cost_original"] = device_clean["maintenance_cost"]

# Flag negative maintenance cost values
device_clean["maintenance_cost_quality_flag"] = np.where(
    device_clean["maintenance_cost"] < 0,
    "Negative Cost Corrected",
    "Valid"
)

# Convert maintenance cost to positive analysis value
#device_clean["maintenance_cost"] = device_clean["maintenance_cost"].abs().round(2)

# Rename downtime to downtime_hours for clarity
device_clean = device_clean.rename(columns={"downtime": "downtime_hours"})
device_clean["downtime_hours"] = device_clean["downtime_hours"].round(2)

# Round financial amount
financial_clean["amount"] = financial_clean["amount"].round(2)

print("Negative maintenance cost records corrected:")
print((device_clean["maintenance_cost_quality_flag"] == "Negative Cost Corrected").sum())



# Step 9: Correct negative maintenance costs

# Count negative maintenance costs BEFORE correction
negative_maintenance_count = (device_clean["maintenance_cost"] < 0).sum()

# Correct negative maintenance costs by converting them to positive values
device_clean["maintenance_cost"] = device_clean["maintenance_cost"].abs()

# Validate that no negative maintenance costs remain
negative_maintenance_remaining = (device_clean["maintenance_cost"] < 0).sum()

print("Negative maintenance costs corrected:", negative_maintenance_count)
print("Negative maintenance costs remaining:", negative_maintenance_remaining)


Negative maintenance cost records corrected:
105
Negative maintenance costs corrected: 105
Negative maintenance costs remaining: 0


In [13]:
# ==========================================
# Add date fields
# ==========================================

device_clean ["purchase_year"]       = device_clean ["purchase_date"].dt.year  
device_clean ["purchase_month"]      = device_clean ["purchase_date"].dt.month   
device_clean ["purchase_month_name"] = device_clean ["purchase_date"].dt.month_name() 

financial_clean["expense_year"]       = financial_clean ["date"].dt.year 
financial_clean["expense_month"]      = financial_clean ["date"].dt.month    
financial_clean["expense_month_name"] = financial_clean ["date"].dt.month_name()   
financial_clean["expense_month_year"] = financial_clean ["date"].dt.to_period("M").astype(str)   

print("Date fields added.") 

Date fields added.


In [15]:
# ==========================================
# Validate device age
# ==========================================

REFERENCE_YEAR = 2025

device_clean ["expected_age"] = REFERENCE_YEAR - device_clean["purchase_year"]

device_clean ["age_validation_flag"] = np.where(

device_clean ["age"] == device_clean ["expected_age"], 

"Matched",

"Review"  

)

print("Age validation results:")

print(device_clean["age_validation_flag"].value_counts())

Age validation results:
age_validation_flag
Matched    4149
Name: count, dtype: int64


In [18]:
# ==========================================
# Create maintenance severity labels
# ==========================================

maintenance_class_map = {
    
    1: "Low",
    2: "Medium",
    3: "High"
}

device_clean["maintenance_severity"] = device_clean["maintenance_class"].map(maintenance_class_map)

print("Maintenance severity counts:")
print(device_clean["maintenance_severity"].value_counts())





Maintenance severity counts:
maintenance_severity
Low       1383
Medium    1383
High      1383
Name: count, dtype: int64


In [19]:
# ==========================================
# Create device age bands
# ==========================================

device_clean["device_age_band"]  = pd.cut(

    device_clean["age"], 

    bins = [ 0 , 3 , 6 , 9, np.inf],   

    labels = [ "0-3 Years", "4-6 Years", "7-9 Years", "10+Years"] 
    
)

print("Device age band counts:")   
print(device_clean["device_age_band"].value_counts().sort_index()) 

Device age band counts:
device_age_band
0-3 Years     863
4-6 Years    1115
7-9 Years    1174
10+Years      997
Name: count, dtype: int64


In [20]:
# ==========================================
# Create failure status flag
# ==========================================


device_clean["failure_status"] = np.where( 

    device_clean["failure_event_count"] > 0,

    "Failure Reported",

    "No Failure Reported" 

)

device_clean["failure_flag"] = np.where(

    device_clean["failure_event_count"] > 0,
    
    1,

    0

)


print("Failure status counts:")
print(device_clean["failure_status"].value_counts())



Failure status counts:
failure_status
Failure Reported       3561
No Failure Reported     588
Name: count, dtype: int64


In [21]:
# ==========================================
# Create failure type from maintenance report text
# ==========================================

def classify_failure_type(report):
    report = str(report).lower()

    if "overheating" in report:
        return "Overheating" 
    elif "battery" in report:
        return "Battery Issue" 
    elif "voltage" in report or "circuit" in report:
        return "Electerical Issue" 
    elif "sensor" in report:
        return"Sensor Issue" 
    elif( 
           "software" in report
        or "driver"   in report
        or "firmware" in report
        or "datalag"  in report
        or "datalog"  in report
        or "crash"    in report
    ):
        return "Software/Data Issue"
    
    elif "abnormal sound" in report:
        return "Mechanical Issue" 
    else:
        return "General Maintenance"  
    
device_clean["failure_type"] = device_clean["maintenance_report"].apply(classify_failure_type)  

print("Failure type counts:")  
print(device_clean["failure_type"].value_counts())   
      

Failure type counts:
failure_type
Electerical Issue      1022
Software/Data Issue     861
Sensor Issue            738
Battery Issue           513
Overheating             488
Mechanical Issue        276
General Maintenance     251
Name: count, dtype: int64


In [22]:
# ==========================================
# Correct financial expense categories
# ==========================================

def correct_expense_category(description, original_category):

    description = str(description).lower()
    
    if "ventilator" in description:
        return "Equipment"
    
    elif "mask" in description:
        return "Supplies"
    
    elif "salary" in description or "salaries" in description or "surgeon" in description:
        return "Staffing"
    
    else:
        return original_category




financial_clean["corrected_expense_category"] = financial_clean.apply(

    lambda row: correct_expense_category(row["description"], row["expense_category"]),

    axis=1
)

financial_clean["expense_category_validation_flag"] = np.where(

    financial_clean["expense_category"] == financial_clean["corrected_expense_category"],
    
    "Matched",
   
    "Corrected"
)

print("Original expense categories:")
print(financial_clean["expense_category"].value_counts())

print("\nCorrected expense categories:")
print(financial_clean["corrected_expense_category"].value_counts())

print("\nCategory validation results:")
print(financial_clean["expense_category_validation_flag"].value_counts())


Original expense categories:
expense_category
Supplies     188
Staffing     167
Equipment    145
Name: count, dtype: int64

Corrected expense categories:
corrected_expense_category
Staffing     183
Supplies     172
Equipment    145
Name: count, dtype: int64

Category validation results:
expense_category_validation_flag
Corrected    350
Matched      150
Name: count, dtype: int64


In [24]:
# ==========================================
# Create operational risk score and risk level
# ==========================================

high_cost_threshold = device_clean["maintenance_cost"].quantile(0.75)
high_downtime_threshold = device_clean["downtime_hours"].quantile(0.75)

device_clean["risk_score"] = 0

device_clean["risk_score"] += np.where(device_clean["age"] >= 8, 1, 0)
device_clean["risk_score"] += np.where(device_clean["maintenance_cost"] >= high_cost_threshold, 1, 0)
device_clean["risk_score"] += np.where(device_clean["downtime_hours"] >= high_downtime_threshold, 1, 0)
device_clean["risk_score"] += np.where(device_clean["failure_event_count"] >= 3, 1, 0)
device_clean["risk_score"] += np.where(device_clean["maintenance_class"] == 3, 1, 0)

def assign_risk_level(score):
    if score >= 4:
        return "High"
    elif score >= 2:
        return "Medium"
    else:
        return "Low"

device_clean["operational_risk_level"] = device_clean["risk_score"].apply(assign_risk_level)

print("High cost threshold:", round(high_cost_threshold, 2))
print("High downtime threshold:", round(high_downtime_threshold, 2))

print("\nOperational risk level counts:")
print(device_clean["operational_risk_level"].value_counts())

High cost threshold: 12023.91
High downtime threshold: 14.64

Operational risk level counts:
operational_risk_level
Low       2292
Medium    1500
High       357
Name: count, dtype: int64


In [25]:
# ==========================================
# Create replacement review flag
# ==========================================

device_clean["replacement_review_flag"] = np.where(
    (
        (device_clean["age"] >= 9) & 
        (device_clean["failure_event_count"] >= 3)
    )
    |
    (
        (device_clean["maintenance_cost"] >= high_cost_threshold) &
        (device_clean["downtime_hours"] >= high_downtime_threshold)
    )
    |
    (
        (device_clean["maintenance_class"] == 3) &
        (device_clean["failure_event_count"] >= 3)
    ),
    "Review for Replacement",
    "Monitor"
)

print("Replacement review flag counts:")
print(device_clean["replacement_review_flag"].value_counts())

Replacement review flag counts:
replacement_review_flag
Monitor                   3095
Review for Replacement    1054
Name: count, dtype: int64


In [26]:
# ==========================================
# Reorder device columns
# ==========================================

device_clean = device_clean[
    [
        "device_id",
        "device_type",
        "purchase_date",
        "purchase_year",
        "purchase_month",
        "purchase_month_name",
        "age",
        "expected_age",
        "age_validation_flag",
        "device_age_band",
        "manufacturer",
        "model",
        "country",
        "maintenance_cost_original",
        "maintenance_cost",
        "maintenance_cost_quality_flag",
        "downtime_hours",
        "maintenance_frequency",
        "failure_event_count",
        "failure_flag",
        "failure_status",
        "maintenance_class",
        "maintenance_severity",
        "maintenance_report",
        "failure_type",
        "risk_score",
        "operational_risk_level",
        "replacement_review_flag"
    ]
]

# ==========================================
# Reorder financial columns
# ==========================================

financial_clean = financial_clean[
    [
        "date",
        "expense_year",
        "expense_month",
        "expense_month_name",
        "expense_month_year",
        "expense_category",
        "corrected_expense_category",
        "expense_category_validation_flag",
        "amount",
        "description"
    ]
]

print("Final column order applied.")

Final column order applied.


In [27]:
# ==========================================
# Final validation checks
# ==========================================

print("Device cleaned shape:", device_clean.shape)
print("Financial cleaned shape:", financial_clean.shape)

print("\nDevice missing values:")
print(device_clean.isnull().sum())

print("\nFinancial missing values:")
print(financial_clean.isnull().sum())

print("\nDevice duplicate rows:", device_clean.duplicated().sum())
print("Financial duplicate rows:", financial_clean.duplicated().sum())

Device cleaned shape: (4149, 28)
Financial cleaned shape: (500, 10)

Device missing values:
device_id                        0
device_type                      0
purchase_date                    0
purchase_year                    0
purchase_month                   0
purchase_month_name              0
age                              0
expected_age                     0
age_validation_flag              0
device_age_band                  0
manufacturer                     0
model                            0
country                          0
maintenance_cost_original        0
maintenance_cost                 0
maintenance_cost_quality_flag    0
downtime_hours                   0
maintenance_frequency            0
failure_event_count              0
failure_flag                     0
failure_status                   0
maintenance_class                0
maintenance_severity             0
maintenance_report               0
failure_type                     0
risk_score                       

In [29]:
# ==========================================
# Create cleaning summary
# ==========================================

cleaning_summary = pd.DataFrame({
    "Metric": [
        "Device Raw Rows",
        "Device Cleaned Rows",
        "Device Raw Columns",
        "Device Cleaned Columns",
        "Device Duplicate Rows After Cleaning",
        "Device Missing Values After Cleaning",
        "Negative Maintenance Costs Corrected",
        "Device Age Records Flagged for Review",
        "Financial Raw Rows",
        "Financial Cleaned Rows",
        "Financial Raw Columns",
        "Financial Cleaned Columns",
        "Financial Duplicate Rows After Cleaning",
        "Financial Missing Values After Cleaning",
        "Financial Expense Categories Corrected"
    ],
    "Value": [
        medical_device_df.shape[0],
        device_clean.shape[0],
        
        medical_device_df.shape[1],
        device_clean.shape[1],
        device_clean.duplicated().sum(),
        device_clean.isnull().sum().sum(),
        (device_clean["maintenance_cost_quality_flag"] == "Negative Cost Corrected").sum(),
        (device_clean["age_validation_flag"] == "Review").sum(),
       
        financial_df.shape[0],
        financial_clean.shape[0],
        financial_df.shape[1],
        financial_clean.shape[1],
        financial_clean.duplicated().sum(),
        financial_clean.isnull().sum().sum(),
        (financial_clean["expense_category_validation_flag"] == "Corrected").sum()
    ]
})

cleaning_summary

,Metric,Value
0,Device Raw Rows,4149
1,Device Cleaned Rows,4149
2,Device Raw Columns,13
3,Device Cleaned Columns,28
4,Device Duplicate Rows After Cleaning,0
5,Device Missing Values After Cleaning,0
6,Negative Maintenance Costs Corrected,105
7,Device Age Records Flagged for Review,0
8,Financial Raw Rows,500
9,Financial Cleaned Rows,500


In [33]:
from pathlib import Path

# ==========================================
# Export cleaned datasets and cleaning summary
# ==========================================

# Define export folders
cleaned_folder = Path(r"C:\Users\Charl\OneDrive\Documents\GitHub\Medical Device Failure & Healthcare Supply Chain Risk BI Project\data\cleaned")
exports_folder = Path(r"C:\Users\Charl\OneDrive\Documents\GitHub\Medical Device Failure & Healthcare Supply Chain Risk BI Project\data\exports")

# Make sure folders exist
cleaned_folder.mkdir(parents=True, exist_ok=True)
exports_folder.mkdir(parents=True, exist_ok=True)

# Export cleaned datasets
device_clean.to_csv(cleaned_folder / "medical_device_failure_cleaned.csv", index=False)
financial_clean.to_csv(cleaned_folder / "financial_data_cleaned.csv", index=False)

# Export cleaning summary
cleaning_summary.to_csv(exports_folder / "cleaning_summary.csv", index=False)

print("Exported files:")
print("medical_device_failure_cleaned.csv")
print("financial_data_cleaned.csv")
print("cleaning_summary.csv")

Exported files:
medical_device_failure_cleaned.csv
financial_data_cleaned.csv
cleaning_summary.csv


In [34]:
# ==========================================
# Preview cleaned datasets
# ==========================================

device_clean.head()


,device_id,device_type,purchase_date,purchase_year,purchase_month,purchase_month_name,age,expected_age,age_validation_flag,device_age_band,manufacturer,model,country,maintenance_cost_original,maintenance_cost,maintenance_cost_quality_flag,downtime_hours,maintenance_frequency,failure_event_count,failure_flag,failure_status,maintenance_class,maintenance_severity,maintenance_report,failure_type,risk_score,operational_risk_level,replacement_review_flag
0,MD03449,Defibrillator,2018-04-23,2018,4,April,7,7,Matched,7-9 Years,CardioSync,Model-100,France,7115.349585,7115.349585,Valid,7.93,3,0,0,No Failure Reported,1,Low,Component component upgrade after detecting ov...,Overheating,0,Low,Monitor
1,MD02024,Infusion Pump,2020-12-10,2020,12,December,5,5,Matched,4-6 Years,MedEquip,Model-650,Italy,7290.780658,7290.780658,Valid,7.84,3,4,1,Failure Reported,2,Medium,battery wear caused operational delay; replace...,Battery Issue,1,Low,Monitor
2,MD04239,MRI Scanner,2023-11-22,2023,11,November,2,2,Matched,0-3 Years,ImagingTech,Model-650,France,5635.521788,5635.521788,Valid,13.91,1,2,1,Failure Reported,3,High,data lag caused operational delay; component u...,General Maintenance,1,Low,Monitor
3,MD00153,Defibrillator,2021-03-03,2021,3,March,4,4,Matched,4-6 Years,RescueTech,Model-450,UK,5001.360188,5001.360188,Valid,29.06,3,1,1,Failure Reported,3,High,Routine check completed; battery wear observed...,Battery Issue,2,Medium,Monitor
4,MD03743,Defibrillator,2019-05-16,2019,5,May,6,6,Matched,4-6 Years,RescueTech,Model-450,Canada,7555.132928,7555.132928,Valid,13.94,4,4,1,Failure Reported,2,Medium,Component inspection after detecting voltage s...,Electerical Issue,1,Low,Monitor


In [35]:
financial_clean.head()

,date,expense_year,expense_month,expense_month_name,expense_month_year,expense_category,corrected_expense_category,expense_category_validation_flag,amount,description
0,2024-10-01,2024,10,October,2024-10,Staffing,Supplies,Corrected,29391.86,Surgical masks
1,2024-10-02,2024,10,October,2024-10,Supplies,Supplies,Matched,47757.71,Surgical masks
2,2024-10-03,2024,10,October,2024-10,Supplies,Equipment,Corrected,43996.60,Ventilators
3,2024-10-04,2024,10,October,2024-10,Supplies,Staffing,Corrected,27908.42,Surgeons' salaries
4,2024-10-05,2024,10,October,2024-10,Equipment,Equipment,Matched,39719.60,Ventilators


## Data Cleaning Summary

The medical device failure dataset was cleaned by standardizing column names, converting purchase dates to datetime format, cleaning text fields, correcting negative maintenance cost values, and creating new business logic fields.

New device analysis fields include device age band, failure status, failure type, maintenance severity, operational risk level, risk score, and replacement review flag.

The financial dataset was cleaned by standardizing column names, converting the date field to datetime format, creating month and year fields, and correcting expense categories based on the expense description.

The cleaned datasets were exported to the `data/cleaned/` folder for use in SQL, Excel, and Power BI.